# Week 5 Final Data Profile and Feasibility Review

I used the FraudShield case pack to check whether the available files can support a reviewer-facing fraud triage workflow. I focused on data quality, evidence traceability, a small feature shortlist, and the limits that would block model training.

## Files Reviewed

I reviewed the AC-1589269 evidence summary and the AC-4471021 wallet import sample. I kept them separate because the account IDs do not match. Treating them as one account would create a bad join and would weaken the review.

In [1]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

data_dir = Path('../data')
outputs_dir = Path('../outputs')
reports_dir = Path('../reports')
outputs_dir.mkdir(exist_ok=True)

evidence = pd.read_csv(data_dir / 'evidence_summary_ac_1589269.csv')
wallet = pd.read_csv(data_dir / 'wallet_import_ac_4471021.csv')
wallet['timestamp'] = pd.to_datetime(wallet['timestamp'], errors='coerce')
wallet['amount'] = pd.to_numeric(wallet['amount'], errors='coerce')
wallet = wallet.sort_values('timestamp').reset_index(drop=True)
wallet['minutes_since_previous'] = wallet['timestamp'].diff().dt.total_seconds().div(60)
wallet['is_card_transaction'] = wallet['payment_brand'].isin(['Visa', 'Mastercard'])
wallet['card_bin_expected'] = wallet['is_card_transaction'] & wallet['type'].eq('DEPOSIT')
wallet['card_bin_missing_when_expected'] = wallet['card_bin_expected'] & wallet['card_bin'].isna()


In [2]:
print('Evidence summary rows/columns:', evidence.shape)
print('Wallet import rows/columns:', wallet.shape)
print()
print('Evidence missing values:')
print(evidence.isna().sum())
print()
print('Wallet missing values:')
print(wallet.isna().sum())


Evidence summary rows/columns: (6, 6)
Wallet import rows/columns: (5, 13)

Evidence missing values:
evidence_id    0
source         0
detail         0
use            0
account_id     0
risk_score     0
dtype: int64

Wallet missing values:
id                                0
account_id                        0
timestamp                         0
type                              0
amount                            0
payment_brand                     0
extra_details                     0
card_bin                          2
chargeback_linked                 0
minutes_since_previous            1
is_card_transaction               0
card_bin_expected                 0
card_bin_missing_when_expected    0
dtype: int64


## Data Quality Checks

I checked the basics first: required columns, duplicate wallet IDs, timestamp parsing, positive amounts, expected card BIN coverage, and whether the two files could be joined safely.

In [3]:
checks = [
    {
        'check': 'Evidence file has required columns',
        'result': set(['evidence_id', 'source', 'detail']).issubset(evidence.columns),
        'note': 'Evidence table can support a traceable register.'
    },
    {
        'check': 'Wallet file has required columns',
        'result': set(['id', 'account_id', 'timestamp', 'type', 'amount', 'payment_brand', 'card_bin']).issubset(wallet.columns),
        'note': 'Wallet sample can support transaction checks.'
    },
    {
        'check': 'No duplicate wallet row IDs',
        'result': int(wallet['id'].duplicated().sum()) == 0,
        'note': f"Duplicate wallet IDs found: {int(wallet['id'].duplicated().sum())}."
    },
    {
        'check': 'Wallet timestamps parse cleanly',
        'result': int(wallet['timestamp'].isna().sum()) == 0,
        'note': f"Unparsed timestamps: {int(wallet['timestamp'].isna().sum())}."
    },
    {
        'check': 'Wallet amounts are positive',
        'result': int((wallet['amount'] <= 0).sum()) == 0,
        'note': f"Non-positive amounts: {int((wallet['amount'] <= 0).sum())}."
    },
    {
        'check': 'Card deposits include card BIN',
        'result': int(wallet['card_bin_missing_when_expected'].sum()) == 0,
        'note': f"Card deposits missing BIN: {int(wallet['card_bin_missing_when_expected'].sum())}."
    },
    {
        'check': 'Files should not be joined as one account',
        'result': 'AC-1589269' != ', '.join(sorted(wallet['account_id'].dropna().unique())),
        'note': f"Evidence case is AC-1589269; wallet import is {', '.join(sorted(wallet['account_id'].dropna().unique()))}."
    },
]
checks_df = pd.DataFrame(checks)
checks_df['status'] = checks_df['result'].map({True: 'Pass', False: 'Review'})
print(checks_df[['check', 'status', 'note']].to_string(index=False))
checks_df.to_csv(outputs_dir / 'data_quality_checks.csv', index=False)


                                    check status                                                      note
       Evidence file has required columns   Pass          Evidence table can support a traceable register.
         Wallet file has required columns   Pass             Wallet sample can support transaction checks.
              No duplicate wallet row IDs   Pass                            Duplicate wallet IDs found: 0.
          Wallet timestamps parse cleanly   Pass                                   Unparsed timestamps: 0.
              Wallet amounts are positive   Pass                                  Non-positive amounts: 0.
           Card deposits include card BIN   Pass                             Card deposits missing BIN: 0.
Files should not be joined as one account   Pass Evidence case is AC-1589269; wallet import is AC-4471021.


## Evidence Metrics

The evidence table stores several useful metrics inside text fields. I extracted those values to see which case-level features could be carried into a first scorecard.

In [4]:
def first_number(pattern, text, default=None, cast=float):
    match = re.search(pattern, text)
    if not match:
        return default
    return cast(match.group(1).replace(',', '').rstrip('.'))

def detail_for(evidence_id):
    rows = evidence.loc[evidence['evidence_id'].eq(evidence_id), 'detail']
    return rows.iloc[0] if len(rows) else ''

wallet_summary = detail_for('ev-wallet-summary')
cash_movement = detail_for('ev-cash-movement')
geo_summary = detail_for('ev-geo-summary')
betting_summary = detail_for('ev-betting-summary')

case_metrics = {
    'case_account_id': 'AC-1589269',
    'wallet_account_id': ', '.join(sorted(wallet['account_id'].dropna().unique())),
    'deposits_count': first_number(r'(\d+) deposits', wallet_summary, 0, int),
    'deposit_total': first_number(r'deposits totaling ([0-9.]+)', wallet_summary, 0.0, float),
    'withdrawals_count': first_number(r'; (\d+) withdrawals', wallet_summary, 0, int),
    'withdrawal_total': first_number(r'withdrawals totaling ([0-9.]+)', wallet_summary, 0.0, float),
    'fast_withdrawals_24h': first_number(r'(\d+) withdrawals occurred within 24 hours', cash_movement, 0, int),
    'withdrawal_deposit_ratio_pct': first_number(r'about (\d+)% of deposits', cash_movement, 0, int),
    'device_count': first_number(r'(\d+) devices', geo_summary, 0, int),
    'ip_count': first_number(r'(\d+) IP', geo_summary, 0, int),
    'city_count': first_number(r'(\d+) cities', geo_summary, 0, int),
    'bet_count': first_number(r'(\d+) bets', betting_summary, 0, int),
    'total_stake': first_number(r'with ([0-9.]+) total stake', betting_summary, 0.0, float),
    'rejected_wagers': first_number(r'and (\d+) rejected wager', betting_summary, 0, int),
}
for key, value in case_metrics.items():
    print(f'{key}: {value}')


case_account_id: AC-1589269
wallet_account_id: AC-4471021
deposits_count: 501
deposit_total: 30577.94
withdrawals_count: 174
withdrawal_total: 19035.22
fast_withdrawals_24h: 163
withdrawal_deposit_ratio_pct: 62
device_count: 5
ip_count: 277
city_count: 7
bet_count: 19936
total_stake: 313143.36
rejected_wagers: 0


## Wallet Sample Read

The wallet sample is small, but it is still useful for checking transaction-level fields and for showing the kind of validation the full system would need.

In [5]:
print('Wallet transaction counts:')
print(wallet['type'].value_counts())
print()
print('Amount summary:')
print(wallet['amount'].describe())
print()
print('Card deposits missing expected BIN:', int(wallet['card_bin_missing_when_expected'].sum()))
print('Chargeback-linked deposits:', int(wallet.loc[wallet['type'].eq('DEPOSIT'), 'chargeback_linked'].sum()))


Wallet transaction counts:
type
DEPOSIT       3
WITHDRAWAL    2
Name: count, dtype: int64

Amount summary:
count       5.000000
mean     3680.000000
std      2193.342199
min       950.000000
25%      1800.000000
50%      4750.000000
75%      4800.000000
max      6100.000000
Name: amount, dtype: float64

Card deposits missing expected BIN: 0
Chargeback-linked deposits: 2


## Data Quality Dashboard

I saved the dashboard as "week5_data_quality_dashboard.png" and "week5_data_quality_dashboard.svg". It shows missingness, the wallet sample, evidence source coverage, and feature readiness.


In [6]:
missing_rows = []
for name, df in [('Evidence summary', evidence), ('Wallet import', wallet)]:
    for column, count in df.isna().sum().items():
        missing_rows.append({
            'file': name,
            'column': column,
            'missing_count': int(count),
            'missing_percent': round((count / len(df)) * 100, 1) if len(df) else 0,
        })
missing_df = pd.DataFrame(missing_rows)
feature_df = pd.read_csv(reports_dir / 'top10_feature_shortlist.csv')

nonzero_missing = missing_df[missing_df['missing_count'] > 0].copy()
if nonzero_missing.empty:
    nonzero_missing = missing_df.head(1).copy()
nonzero_missing['label'] = nonzero_missing['file'] + ' | ' + nonzero_missing['column']
source_counts = evidence['source'].value_counts().sort_values()
readiness_counts = feature_df['current_support'].value_counts().sort_values()

green = '#3f7252'
copper = '#a4653b'
gold = '#988550'
grey = '#676767'
grid = '#dddddd'

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('FraudShield Week 5 Data Quality Dashboard', fontsize=20, fontweight='bold', y=0.98)

ax = axes[0, 0]
ax.barh(nonzero_missing['label'], nonzero_missing['missing_count'], color=copper)
for idx, row in nonzero_missing.reset_index(drop=True).iterrows():
    ax.text(row['missing_count'] + 0.05, idx, f"{row['missing_count']} ({row['missing_percent']}%)", va='center')
ax.set_title('Missing fields that need interpretation')
ax.set_xlabel('Missing count')
ax.set_xlim(0, max(nonzero_missing['missing_count'].max() + 1, 3))
ax.grid(axis='x', alpha=0.35, color=grid)

ax = axes[0, 1]
for tx_type, color in [('DEPOSIT', green), ('WITHDRAWAL', copper)]:
    part = wallet[wallet['type'].eq(tx_type)].copy()
    part['minutes_from_start'] = part['timestamp'].sub(wallet['timestamp'].min()).dt.total_seconds().div(60)
    ax.scatter(part['minutes_from_start'], part['amount'], s=95, color=color, label=tx_type, edgecolor='#333333', linewidth=0.7, alpha=0.9)
chargeback_count = int(wallet['chargeback_linked'].sum())
if chargeback_count:
    cb = wallet[wallet['chargeback_linked']].copy()
    cb['minutes_from_start'] = cb['timestamp'].sub(wallet['timestamp'].min()).dt.total_seconds().div(60)
    ax.scatter(cb['minutes_from_start'], cb['amount'], s=180, facecolors='none', edgecolors=copper, linewidth=2.0, label='Chargeback linked')
ax.set_title(f'Wallet sample, {chargeback_count} chargeback-linked deposits')
ax.set_xlabel('Minutes from first transaction')
ax.set_ylabel('Amount')
ax.legend(frameon=False, loc='upper right')
ax.grid(alpha=0.3, color=grid)

ax = axes[1, 0]
ax.barh(source_counts.index, source_counts.values, color=grey)
for idx, value in enumerate(source_counts.values):
    ax.text(value + 0.05, idx, str(value), va='center')
ax.set_title('Evidence rows by source table')
ax.set_xlabel('Evidence rows')
ax.set_xlim(0, source_counts.max() + 1)
ax.grid(axis='x', alpha=0.35, color=grid)

ax = axes[1, 1]
ready_colors = [green if label == 'Ready now' else gold if 'display' in label else copper if 'KYC' in label else grey for label in readiness_counts.index]
ax.barh(readiness_counts.index, readiness_counts.values, color=ready_colors)
for idx, value in enumerate(readiness_counts.values):
    ax.text(value + 0.05, idx, str(value), va='center')
ax.set_title('Top-10 feature readiness')
ax.set_xlabel('Feature count')
ax.set_xlim(0, readiness_counts.max() + 1)
ax.grid(axis='x', alpha=0.35, color=grid)

for ax in axes.ravel():
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.95])
png_path = outputs_dir / 'week5_data_quality_dashboard.png'
svg_path = outputs_dir / 'week5_data_quality_dashboard.svg'
fig.savefig(png_path, dpi=180, bbox_inches='tight')
fig.savefig(svg_path, bbox_inches='tight')
plt.close(fig)
print(missing_df.to_string(index=False))


            file                         column  missing_count  missing_percent
Evidence summary                    evidence_id              0              0.0
Evidence summary                         source              0              0.0
Evidence summary                         detail              0              0.0
Evidence summary                            use              0              0.0
Evidence summary                     account_id              0              0.0
Evidence summary                     risk_score              0              0.0
   Wallet import                             id              0              0.0
   Wallet import                     account_id              0              0.0
   Wallet import                      timestamp              0              0.0
   Wallet import                           type              0              0.0
   Wallet import                         amount              0              0.0
   Wallet import                  paymen

![FraudShield Week 5 data quality dashboard](../outputs/week5_data_quality_dashboard.svg)


## Top-10 Feature Shortlist

I selected features that a reviewer could understand and trace back to the case pack. I did not shortlist features that would require guessing from missing tables.

In [7]:
feature_df = pd.read_csv(reports_dir / 'top10_feature_shortlist.csv')
print(feature_df[['feature', 'source', 'current_support']].to_string(index=False))


                           feature           source                            current_support
       account_status_under_review Evidence summary                                  Ready now
                kyc_status_unknown     Analyst note                 Needs structured KYC field
                        risk_score     Analyst note Ready for display, not enough for training
                     deposit_total Evidence summary                                  Ready now
                  withdrawal_total Evidence summary                                  Ready now
       withdrawal_to_deposit_ratio Evidence summary                                  Ready now
              fast_withdrawals_24h Evidence summary                                  Ready now
        payment_instrument_pattern    Wallet import                             Prototype only
             device_ip_city_spread Evidence summary                                  Ready now
betting_volume_and_rejected_wagers Evidence summar

## Final Feasibility Read

The current files can support a narrow analyst decision-support prototype: profile the case, show evidence IDs, flag data gaps, and recommend enhanced KYC or source-of-funds review. They cannot support meaningful model training yet. The sample is too small, key tables are missing, and the wallet import belongs to a different account from the evidence summary.